# Data Setup

In [1]:
### Setup Data Module for Lightning
import lightning as L
from torch.utils.data import DataLoader, ConcatDataset
from rastervision.pytorch_learner.object_detection_utils import collate_fn as od_collate

class ChacuDataModule(L.LightningDataModule):
    def __init__(self, train_ds_list, val_ds_list, batch_size=8, num_workers=4):
        super().__init__()
        self.train_ds_list = train_ds_list
        self.val_ds_list = val_ds_list
        self.batch_size = batch_size
        self.num_workers = num_workers
        
        self.train_dataset = None
        self.val_dataset = None

    def setup(self, stage=None):
        # Combine the lists of datasets into single ConcatDatasets
        if stage == "fit" or stage is None:
            self.train_dataset = ConcatDataset(self.train_ds_list)
            self.val_dataset = ConcatDataset(self.val_ds_list)
        
        if stage == "test":
            self.val_dataset = ConcatDataset(self.val_ds_list)

    def train_dataloader(self):
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            collate_fn=od_collate,  # Crucial for Raster Vision OD datasets
            pin_memory=True
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            collate_fn=od_collate,
            pin_memory=True
        )

/home/VANDERBILT/zimmejr1/anaconda3/envs/DinoV3/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-01-08 11:41:18:rastervision.pipeline.rv_config: WARNING - Root temporary directory cannot be used: /opt/data/tmp. Using root: /tmp/tmp0mrdkl19


In [2]:
#set data paths and configs
import os
import geopandas as gpd

import pathlib

from rastervision.core.data import GeoJSONVectorSource, RasterioCRSTransformer,ClassConfig
from rastervision.pytorch_learner import ClassificationSlidingWindowGeoDataset
from rastervision.core.data import RasterioSource
from geopacha_utilities.utilities import find_pixel_size
TRAINING_AOI_DIRECTORY = '/mnt/sarl_commons06/Wernke_projects/zimmejr1/DinoV3_Chacu_AOI_12_23_25/training_aoi'
VALIDATION_AOI_DIRECTORY = '/mnt/sarl_commons06/Wernke_projects/zimmejr1/DinoV3_Chacu_AOI_12_23_25/validation_aoi_sampled'
LABEL_URI = '/mnt/sarl_commons06/Wernke_projects/zimmejr1/DinoV3_Chacu_AOI_12_23_25/chacu_labels.geojson'
IMAGERY_BASE_DIRECTORY = '/mnt/sarl_commons06/Wernke_projects/GeoPACHA/Imagery_Machine_Learning/Analysis_Images'

patch_dim = 1024

In [3]:
import albumentations as A

data_augmentation_transform = A.Compose([
    A.D4(p=1.0),
])

In [4]:
from tqdm import tqdm
from rastervision.pytorch_learner import ObjectDetectionRandomWindowGeoDataset,ObjectDetectionSlidingWindowGeoDataset
from rastervision.pytorch_learner.object_detection_utils import collate_fn as od_collate
from rastervision.core.data import ObjectDetectionLabelSourceConfig
from rastervision.core.data import ObjectDetectionLabelSource

class_config = ClassConfig(
    names=['chacu', 'background'],
    colors=['darkred', 'gray'],
    null_class='background')



training_dataset_list = []
val_dataset_list = []
for aoi_path in tqdm(os.listdir(TRAINING_AOI_DIRECTORY),desc="making datasets"):
        full_aoi_path = os.path.join(TRAINING_AOI_DIRECTORY,aoi_path)
        aoi = gpd.read_file(full_aoi_path)
        image_id = aoi['imageid'][0]
        # print(image_id)
        image_path  = pathlib.PureWindowsPath(aoi['filepath'][0]).as_posix()
        full_image_path = os.path.join(IMAGERY_BASE_DIRECTORY,image_path)

        #Adjust the patch size to account for different resolution
        rasterSource = RasterioSource(
                full_image_path, #path to the image
                allow_streaming=True, # allow_streaming so we don't have to load the whole image
            ) 
        pixel_size = find_pixel_size(rasterSource.imagery_path)
        size = round(patch_dim*.5/pixel_size)



        # print("CREATING DATASET")
        try:
            ds = ObjectDetectionRandomWindowGeoDataset.from_uris(
                image_uri=full_image_path,
                aoi_uri=full_aoi_path,
                label_vector_uri = LABEL_URI,
                class_config=class_config,
                image_raster_source_kw=dict(allow_streaming=True,channel_order=[4,2,1]),
                max_windows=4,
                size_lims = [size,size+1],
                out_size=patch_dim,within_aoi=True,ioa_thresh = 0.75,neg_ratio=.5,
                transform = data_augmentation_transform
            )
            ds.scene.id=image_id
            training_dataset_list.append(ds)
        except:
            try:
                # print("Extracting Negatives")
                ds = ObjectDetectionRandomWindowGeoDataset.from_uris(
                    image_uri=full_image_path,
                    aoi_uri=full_aoi_path,
                    label_vector_uri = LABEL_URI,
                    class_config=class_config,
                    image_raster_source_kw=dict(allow_streaming=True,channel_order=[4,2,1]),
                    max_windows=2,
                    size_lims = [size,size+1],
                    out_size=patch_dim,within_aoi=False,
                    transform = data_augmentation_transform
                )
                ds.scene.id=image_id
                training_dataset_list.append(ds)
            except Exception as e: 
                print(f"Couldn't create dataset because:\n{e}")                   
                continue
print("MAKING VALIDATION DATA")
for aoi_path in tqdm(os.listdir(VALIDATION_AOI_DIRECTORY),desc="making datasets"):
        full_aoi_path = os.path.join(VALIDATION_AOI_DIRECTORY,aoi_path)
        aoi = gpd.read_file(full_aoi_path)
        image_id = aoi['imageid'][0]
        # print(image_id)
        image_path  = pathlib.PureWindowsPath(aoi['filepath'][0]).as_posix()
        full_image_path = os.path.join(IMAGERY_BASE_DIRECTORY,image_path)

        #Adjust the patch size to account for different resolution
        rasterSource = RasterioSource(
                full_image_path, #path to the image
                allow_streaming=True, # allow_streaming so we don't have to load the whole image
            ) 
        pixel_size = find_pixel_size(rasterSource.imagery_path)
        size = round(patch_dim*.5/pixel_size)


        try:
              ds = ObjectDetectionSlidingWindowGeoDataset.from_uris(
                    image_uri = full_image_path,
                    aoi_uri = full_aoi_path,
                    label_vector_uri = LABEL_URI,
                    class_config = class_config,
                    size = size,
                    stride = int(size*0.5),
                    out_size = patch_dim,within_aoi=True,
                    image_raster_source_kw=dict(allow_streaming=True,channel_order=[4,2,1]),return_window=False

              )
              ds.scene.id = image_id
              val_dataset_list.append(ds)
        except Exception as e:
              print(f"Problem with {image_id} sliding window: \n {e}")
              continue

making datasets: 100%|██████████| 118/118 [00:44<00:00,  2.65it/s]


MAKING VALIDATION DATA


making datasets: 100%|██████████| 6/6 [00:08<00:00,  1.44s/it]


In [5]:
# 1. Initialize the DataModule
dm = ChacuDataModule(
    train_ds_list=training_dataset_list, 
    val_ds_list=val_dataset_list, 
    batch_size=8, # Start small to ensure no OOM on 1024px patches
    num_workers=8
)

# Model Setup

In [6]:
import torch

# This is the standard "sweet spot" for most deep learning tasks
torch.set_float32_matmul_precision('high')

In [ ]:
### Setup Model Module for Lightning
import lightning as L
from torch.optim import AdamW
from rastervision.pytorch_learner.object_detection_utils import compute_coco_eval

class ChacuDetectionModule(L.LightningModule):
    def __init__(self, model, class_config, lr=5e-5):
        super().__init__()
        self.model = model  # model should be the TorchVisionODAdapter(raw_model)
        self.lr = lr
        self.class_config = class_config
        self.validation_step_outputs = []
        self.training_step_outputs = []
    
    def to_device(self, x, device):
        """Replicating the Raster Vision helper to handle BoxLists/Tensors."""
        if isinstance(x, list):
            return [_x.to(device) if _x is not None else _x for _x in x]
        return x.to(device)
    
    def forward(self, x):
        # When you call model(x), it will now execute this:
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, targets = batch
        loss_dict = self.model(images, targets)
        total_loss = sum(loss_dict.values())
        
        # 1. Prepare log dict (detach to save memory)
        log_vars = {k: v.detach().cpu() for k, v in loss_dict.items()}
        log_vars['train_loss'] = total_loss.detach().cpu()
        
        # 2. Append to our list
        self.training_step_outputs.append(log_vars)
        
        # 3. Log to progress bar only (so you see it while training)
        self.log("loss", total_loss, prog_bar=True, on_step=True, on_epoch=False,logger=False)
        
        return total_loss
    
    def on_train_epoch_end(self):
        if not self.training_step_outputs:
            return

        # 1. Aggregate and average across the ~47 steps
        keys = self.training_step_outputs[0].keys()
        avg_losses = {}
        for k in keys:
            avg_losses[k] = torch.stack([x[k] for x in self.training_step_outputs]).mean()

        # 2. Log with manual step for RV parity (0, 1, 2...)
        self.logger.log_metrics(avg_losses, step=self.current_epoch)

        # 3. Clear for the next epoch
        self.training_step_outputs.clear()

    def validation_step(self, batch, batch_idx):
        images, targets = batch
        outputs = self.model(images)
        
        # Use the new helper to move everything to CPU for evaluation
        res = {
            'ys': self.to_device(targets, 'cpu'), 
            'outs': self.to_device(outputs, 'cpu')
        }
        
        self.validation_step_outputs.append(res)
        return res

    def on_validation_epoch_end(self):
        # Flatten and compute COCO metrics
        all_ys = []
        all_outs = []
        for out in self.validation_step_outputs:
            all_ys.extend(out['ys'])
            all_outs.extend(out['outs'])

        num_class_ids = len(self.class_config.names)
        coco_eval = compute_coco_eval(all_outs, all_ys, num_class_ids)

        if coco_eval is not None:
            self.logger.log_metrics({
                "mAP50": coco_eval.stats[1],
                "mAP": coco_eval.stats[0]
            }, step=self.current_epoch)

        self.validation_step_outputs.clear()

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.lr)
        # We can monitor "train_loss" or "mAP50" for the scheduler
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, 'max', patience=5, factor=0.5
        )
        return {
            "optimizer": optimizer,
            # "lr_scheduler": {"scheduler": scheduler, "monitor": "mAP50"},
        }

In [8]:
# # Initialize model
# from src.detection.model import dinov3_detection
# repo_path = "/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3"
# weights_path = "/home/VANDERBILT/zimmejr1/Documents/DinoV3_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
# model_name  = "dinov3_vitl16"
# model = dinov3_detection(
#     fine_tune=True,
#     num_classes=2, 
#     weights=weights_path,
#     model_name=model_name,
#     repo_dir=repo_path,
#     feature_extractor="multi",
#     head="retinanet"
# )

# # 2. Initialize the Model (Assuming 'raw_model' is your DinoV3 architecture)
# # Important: Ensure adapter_model is what goes into the LightningModule
# from rastervision.pytorch_learner import TorchVisionODAdapter
# adapter_model = TorchVisionODAdapter(model) 

# model = ChacuDetectionModule(
#     model=adapter_model, 
#     class_config=class_config, 
#     lr=5e-5
# )

In [9]:
import torch
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.ops import sigmoid_focal_loss
from src.detection.model import dinov3_detection
repo_path = "/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3"
weights_path = "/home/VANDERBILT/zimmejr1/Documents/DinoV3_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
model_name  = "dinov3_vitl16"


def _sum(x):
    res = x[0]
    for i in x[1:]:
        res = res + i
    return res
# 1. Define the custom loss logic inside a subclass of the Head
class CalibratedRetinaNetHead(torch.nn.Module):
    def __init__(self, original_head, smoothing, gamma):
        super().__init__()
        self.original_head = original_head
        self.smoothing = smoothing
        self.gamma = gamma

        # This is to fix using det_utils.Matcher.BETWEEN_THRESHOLDS in TorchScript.
        # TorchScript doesn't support class attributes.
        # https://github.com/pytorch/vision/pull/1697#issuecomment-630255584
        self.BETWEEN_THRESHOLDS = -2

    def forward(self, x):
        return self.original_head(x)

    
    def compute_loss(self, targets, head_outputs, matched_idxs):
        losses = []

        cls_logits = head_outputs["cls_logits"]

        for targets_per_image, cls_logits_per_image, matched_idxs_per_image in zip(targets, cls_logits, matched_idxs):
            # determine only the foreground
            foreground_idxs_per_image = matched_idxs_per_image >= 0
            num_foreground = foreground_idxs_per_image.sum()

            # create the target classification
            gt_classes_target = torch.zeros_like(cls_logits_per_image)
            gt_classes_target[
                foreground_idxs_per_image,
                targets_per_image["labels"][matched_idxs_per_image[foreground_idxs_per_image]],
            ] = 1.0

            # --- Apply Label Smoothing ---
            # Formula: target = target * (1 - smoothing) + 0.5 * smoothing
            # This pushes 0.0 to epsilon and 1.0 to 1-epsilon
            if hasattr(self, 'smoothing') and self.smoothing > 0:
                gt_classes_target = gt_classes_target * (1 - self.smoothing) + 0.5 * self.smoothing

            # find indices for which anchors should be ignored
            valid_idxs_per_image = matched_idxs_per_image != self.BETWEEN_THRESHOLDS

            # compute the classification loss
            losses.append(
                sigmoid_focal_loss(
                    cls_logits_per_image[valid_idxs_per_image],
                    gt_classes_target[valid_idxs_per_image],
                    reduction="sum",
                    gamma=self.gamma
                )
                / max(1, num_foreground)
            )

        return _sum(losses) / len(targets)
    
    # 2. Function to "Patch" your model for Raster Vision
def get_calibrated_dinov3_model(num_classes, smoothing, gamma):
    # This is where you'd initialize your DINOv3 + RetinaNet model
    model = dinov3_detection(
        fine_tune=True,
        num_classes=2, 
        weights=weights_path,
        model_name=model_name,
        repo_dir=repo_path,
        feature_extractor="multi",
        head="retinanet"
    ) 
    
    # Wrap the existing classification head with our calibrated version
    original_head = model.head.classification_head
    model.head.classification_head = CalibratedRetinaNetHead(
        original_head, 
        smoothing=smoothing, 
        gamma=gamma
    )
    return model

model = get_calibrated_dinov3_model(num_classes=2,smoothing=0.01,gamma=2.5)
from rastervision.pytorch_learner.object_detection_utils import TorchVisionODAdapter
adapter_model = TorchVisionODAdapter(model)

model = ChacuDetectionModule(
    model=adapter_model, 
    class_config=class_config, 
    lr=5e-5
)

Loading pretrained backbone weights from:  /home/VANDERBILT/zimmejr1/Documents/DinoV3_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth


In [10]:
from lightning.pytorch.loggers import TensorBoardLogger, CSVLogger

# Define the logger
# This will create a folder structure: logs/chacu_dinov3/version_0/
tb_logger = TensorBoardLogger(
    save_dir="data", 
    name="chacu_dinov3_lightning_test_calibrated"
)
csv_logger = CSVLogger("logs", name="chacu_experiment")

In [11]:
import torch
from lightning.pytorch.callbacks import Callback
from torchvision.utils import draw_bounding_boxes

class DetectionVisualizer(Callback):
    def __init__(self, num_samples=3):
        super().__init__()
        self.num_samples = num_samples

    def on_validation_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        # Only capture samples from the first batch of validation
        if batch_idx == 0:
            images, targets = batch
            # Switch model to eval mode for predictions
            pl_module.eval()
            with torch.no_grad():
                preds = pl_module.model(images)
            
            for i in range(min(len(images), self.num_samples)):
                # 1. Prepare the image (un-normalize if necessary)
                # Assuming images are 0-1 float tensors
                img_uint8 = (images[i] * 255).to(torch.uint8).cpu()
                
                # 2. Draw Ground Truth (Green)
                gt_boxes = targets[i]['boxes'].cpu()
                img_with_boxes = draw_bounding_boxes(img_uint8, gt_boxes, colors="green", width=2)
                
                # 3. Draw Predictions (Red) - Filter by a confidence threshold
                pred_boxes = preds[i]['boxes'].cpu()
                pred_scores = preds[i]['scores'].cpu()
                high_conf_indices = pred_scores > 0.3
                
                final_img = draw_bounding_boxes(
                    img_with_boxes, 
                    pred_boxes[high_conf_indices], 
                    colors="red", 
                    width=2
                )
                
                # 4. Log to TensorBoard
                trainer.logger.experiment.add_image(
                    f"Validation_Sample_{i}", 
                    final_img, 
                    global_step=trainer.global_step
                )

In [12]:
# visualizer = DetectionVisualizer(num_samples=4)

In [ ]:
from lightning.pytorch.callbacks import TQDMProgressBar
progress_bar = TQDMProgressBar(refresh_rate=2)
trainer = L.Trainer(
    max_epochs=100,
    accelerator="gpu",
    devices=2,
    precision="16-mixed",
    logger=[tb_logger,csv_logger],  # Connect the logger here
    log_every_n_steps=50,  # How often to log training_loss
    callbacks = [progress_bar]
)

In [14]:



# 3. Start Training
trainer.fit(model, datamodule=dm)

In [14]:


#Custom Packages
from src.detection.model import dinov3_detection
from geopacha_utilities.utilities import find_pixel_size
def _sum(x):
    res = x[0]
    for i in x[1:]:
        res = res + i
    return res
class CalibratedRetinaNetHead(torch.nn.Module):
    def __init__(self, original_head, smoothing, gamma):
        super().__init__()
        self.original_head = original_head
        self.smoothing = smoothing
        self.gamma = gamma

        # This is to fix using det_utils.Matcher.BETWEEN_THRESHOLDS in TorchScript.
        # TorchScript doesn't support class attributes.
        # https://github.com/pytorch/vision/pull/1697#issuecomment-630255584
        self.BETWEEN_THRESHOLDS = -2

    def forward(self, x):
        return self.original_head(x)

    
    def compute_loss(self, targets, head_outputs, matched_idxs):
        losses = []

        cls_logits = head_outputs["cls_logits"]

        for targets_per_image, cls_logits_per_image, matched_idxs_per_image in zip(targets, cls_logits, matched_idxs):
            # determine only the foreground
            foreground_idxs_per_image = matched_idxs_per_image >= 0
            num_foreground = foreground_idxs_per_image.sum()

            # create the target classification
            gt_classes_target = torch.zeros_like(cls_logits_per_image)
            gt_classes_target[
                foreground_idxs_per_image,
                targets_per_image["labels"][matched_idxs_per_image[foreground_idxs_per_image]],
            ] = 1.0

            # --- Apply Label Smoothing ---
            # Formula: target = target * (1 - smoothing) + 0.5 * smoothing
            # This pushes 0.0 to epsilon and 1.0 to 1-epsilon
            if hasattr(self, 'smoothing') and self.smoothing > 0:
                gt_classes_target = gt_classes_target * (1 - self.smoothing) + 0.5 * self.smoothing

            # find indices for which anchors should be ignored
            valid_idxs_per_image = matched_idxs_per_image != self.BETWEEN_THRESHOLDS

            # compute the classification loss
            losses.append(
                sigmoid_focal_loss(
                    cls_logits_per_image[valid_idxs_per_image],
                    gt_classes_target[valid_idxs_per_image],
                    reduction="sum",
                    gamma=self.gamma
                )
                / max(1, num_foreground)
            )

        return _sum(losses) / len(targets)
    
    # 2. Function to "Patch" your model for Raster Vision
def get_calibrated_dinov3_model(num_classes, smoothing, gamma):
    # This is where you'd initialize your DINOv3 + RetinaNet model
    model = dinov3_detection(
        fine_tune=True,
        num_classes=2, 
        weights=weights_path,
        model_name=model_name,
        repo_dir=repo_path,
        feature_extractor="multi",
        head="retinanet"
    ) 
    
    # Wrap the existing classification head with our calibrated version
    original_head = model.head.classification_head
    model.head.classification_head = CalibratedRetinaNetHead(
        original_head, 
        smoothing=smoothing, 
        gamma=gamma
    )
    return model
repo_path = "/home/VANDERBILT/zimmejr1/Documents/GitHub/dinov3"
weights_path = "/home/VANDERBILT/zimmejr1/Documents/DinoV3_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth"
model_name  = "dinov3_vitl16"

In [104]:
model

TorchVisionODAdapter(
  (model): RetinaNet(
    (backbone): Dinov3Backbone(
      (backbone_model): DinoVisionTransformer(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
          (norm): Identity()
        )
        (rope_embed): RopePositionEmbedding()
        (blocks): ModuleList(
          (0-23): 24 x SelfAttentionBlock(
            (norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (attn): SelfAttention(
              (qkv): LinearKMaskedBias(in_features=1024, out_features=3072, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=1024, out_features=1024, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
            )
            (ls1): LayerScale()
            (norm2): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=1024, out_features=4096, bias=True)


In [105]:

model = get_calibrated_dinov3_model(2,0.01,2.0)
checkpoint=torch.load("checkpoints/chacu-dinov3-epoch=31-mAP50=0.539.ckpt",map_location="cpu")
model.load_state_dict(checkpoint['state_dict'])

Loading pretrained backbone weights from:  /home/VANDERBILT/zimmejr1/Documents/DinoV3_weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth


RuntimeError: Error(s) in loading state_dict for RetinaNet:
	Missing key(s) in state_dict: "backbone.backbone_model.cls_token", "backbone.backbone_model.storage_tokens", "backbone.backbone_model.mask_token", "backbone.backbone_model.patch_embed.proj.weight", "backbone.backbone_model.patch_embed.proj.bias", "backbone.backbone_model.rope_embed.periods", "backbone.backbone_model.blocks.0.norm1.weight", "backbone.backbone_model.blocks.0.norm1.bias", "backbone.backbone_model.blocks.0.attn.qkv.weight", "backbone.backbone_model.blocks.0.attn.qkv.bias", "backbone.backbone_model.blocks.0.attn.qkv.bias_mask", "backbone.backbone_model.blocks.0.attn.proj.weight", "backbone.backbone_model.blocks.0.attn.proj.bias", "backbone.backbone_model.blocks.0.ls1.gamma", "backbone.backbone_model.blocks.0.norm2.weight", "backbone.backbone_model.blocks.0.norm2.bias", "backbone.backbone_model.blocks.0.mlp.fc1.weight", "backbone.backbone_model.blocks.0.mlp.fc1.bias", "backbone.backbone_model.blocks.0.mlp.fc2.weigh

In [38]:
for i in range(len(inference_ds)):
    print(inference_ds[i][1])

{'boxes': tensor([], size=(0, 4)), 'class_ids': tensor([], dtype=torch.int64)}
{'boxes': tensor([], size=(0, 4)), 'class_ids': tensor([], dtype=torch.int64)}
{'boxes': tensor([], size=(0, 4)), 'class_ids': tensor([], dtype=torch.int64)}
{'boxes': tensor([], size=(0, 4)), 'class_ids': tensor([], dtype=torch.int64)}
{'boxes': tensor([], size=(0, 4)), 'class_ids': tensor([], dtype=torch.int64)}
{'boxes': tensor([[925.2020, 547.5056, 991.0673, 609.2543]]),
 'class_ids': tensor([0])}
{'boxes': tensor([[413.7166, 547.5056, 479.5819, 609.2543]]),
 'class_ids': tensor([0])}
{'boxes': tensor([], size=(0, 4)), 'class_ids': tensor([], dtype=torch.int64)}
{'boxes': tensor([], size=(0, 4)), 'class_ids': tensor([], dtype=torch.int64)}
{'boxes': tensor([[925.2020,  36.0201, 991.0673,  97.7688]]),
 'class_ids': tensor([0])}
{'boxes': tensor([[413.7166,  36.0201, 479.5819,  97.7688]]),
 'class_ids': tensor([0])}
{'boxes': tensor([], size=(0, 4)), 'class_ids': tensor([], dtype=torch.int64)}
{'boxes': te

KeyboardInterrupt: 

In [84]:
from tqdm.autonotebook import tqdm
import torch
from torch.utils.data import ConcatDataset, DataLoader
from rastervision.core.data import ObjectDetectionLabels
from rastervision.pytorch_learner.object_detection_utils import collate_fn as od_collate
inference_ds = val_dataset_list[1]
inference_dl = DataLoader(
    inference_ds,
    batch_size=1, 
    shuffle=False,     # Never shuffle during inference
    num_workers=8,     # Parallel loading
    collate_fn=od_collate
)
def get_predictions(dataloader, model, device='cuda'):
    model.eval()
    model.to(device)
    
    for x, _ in tqdm(dataloader):
        with torch.inference_mode():
            # Move list of images to GPU
            x = [img.to(device) for img in x]
            
            # The Adapter returns a list of BoxList objects
            out_batch = model(x)
            
        # Yield each BoxList moved to CPU
        for out in out_batch:
            # Convert the BoxList object to a CPU dictionary
            # Raster Vision's ObjectDetectionLabels expects:
            # {'boxes': np.array, 'class_ids': np.array, 'scores': np.array}
                yield {
                    'boxes': out.boxes.cpu().numpy(),
                    'class_ids': out.get_field('class_ids').cpu().numpy(),
                    'scores': out.get_field('scores').cpu().numpy()
                }

# 1. Generate predictions
model.eval()
predictions = get_predictions(inference_dl, model)

# 2. Create the Scene-Level Labels object
# NOTE: Unlike segmentation, we don't use 'smooth' or 'num_classes' here.
pred_labels = ObjectDetectionLabels.from_predictions(
    inference_ds.windows,
    predictions
)

# 3. Non-Maximum Suppression (NMS)
# This is the "Object Detection" version of smoothing. It merges 
# overlapping boxes found at the edges of your sliding windows.
pred_labels = ObjectDetectionLabels.prune_duplicates(
    pred_labels, 
    merge_thresh=0.1, # IoU threshold for merging
    score_thresh=0.5  # Minimum confidence to keep a kiln detection
)

100%|██████████| 64/64 [00:06<00:00,  9.22it/s]


In [94]:
crs_transformer=inference_ds.scene.raster_source.crs_transformer
for window in inference_ds.windows:
    print(crs_transformer.pixel_to_map(window))

Box(ymin=-16.902827425870328, xmin=-70.26843532768339, ymax=-16.907246632466467, xmax=-70.26401612108725)
Box(ymin=-16.902827425870328, xmin=-70.26622572438532, ymax=-16.907246632466467, xmax=-70.26180651778918)
Box(ymin=-16.902827425870328, xmin=-70.26401612108725, ymax=-16.907246632466467, xmax=-70.25959691449111)
Box(ymin=-16.902827425870328, xmin=-70.26180651778918, ymax=-16.907246632466467, xmax=-70.25738731119304)
Box(ymin=-16.905037029168398, xmin=-70.26843532768339, ymax=-16.909456235764537, xmax=-70.26401612108725)
Box(ymin=-16.905037029168398, xmin=-70.26622572438532, ymax=-16.909456235764537, xmax=-70.26180651778918)
Box(ymin=-16.905037029168398, xmin=-70.26401612108725, ymax=-16.909456235764537, xmax=-70.25959691449111)
Box(ymin=-16.905037029168398, xmin=-70.26180651778918, ymax=-16.909456235764537, xmax=-70.25738731119304)
Box(ymin=-16.907246632466467, xmin=-70.26843532768339, ymax=-16.911665839062607, xmax=-70.26401612108725)
Box(ymin=-16.907246632466467, xmin=-70.2662257

In [85]:
pred_labels.save("test-lightning-predict.geojson",class_config=class_config,crs_transformer=inference_ds.scene.raster_source.crs_transformer)

2026-01-08 12:20:03:rastervision.core.data.label_store.object_detection_geojson_store: INFO - Saving 2 boxes as GeoJSON.


In [101]:
output=model(inference_ds[0][0].unsqueeze(0).to('cuda'))

In [103]:
output[0].boxes

tensor([[ 934.2848,    5.7206, 1013.5978,   51.3284]], device='cuda:0',
       grad_fn=<IndexBackward0>)